# Hyperparameter Tuning with Ray Tune

This notebook demonstrates how to perform distributed hyperparameter optimization using Ray Tune. You'll learn how to:

- Define search spaces for hyperparameters
- Use different search algorithms (grid, random, Bayesian)
- Apply schedulers for early stopping (ASHA, PBT)
- Analyze and visualize tuning results
- Integrate with Ray Train for distributed trials

## Prerequisites

- 2+ GPUs (for parallel trials)
- Python 3.9+

## 1. Installation

In [ ]:
!pip install -q "ray[tune]" torch torchvision optuna matplotlib pandas

## 2. Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")

from utils import (
    print_gpu_status,
    detect_gpus,
    init_ray,
    shutdown_ray,
    ClusterMode,
    print_best_trial,
    plot_tune_results,
)

In [ ]:
print_gpu_status()

gpu_info = detect_gpus()
NUM_GPUS = gpu_info["count"] if gpu_info["available"] else 0
print(f"\nAvailable GPUs for parallel trials: {NUM_GPUS}")

In [ ]:
# Configuration
CLUSTER_MODE = ClusterMode.LOCAL
STORAGE_PATH = "./runs/hyperparameter_tuning"

# Number of parallel trials (limited by GPUs)
MAX_CONCURRENT_TRIALS = max(1, NUM_GPUS)

In [ ]:
init_ray(mode=CLUSTER_MODE)

## 3. Define Training Function

The training function receives hyperparameters from Ray Tune and reports metrics back.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from ray import tune, train
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch


def train_cifar(config):
    """
    Training function for hyperparameter tuning.
    
    Args:
        config: Dictionary of hyperparameters from Ray Tune
    """
    # Build model with tunable architecture
    model = nn.Sequential(
        nn.Conv2d(3, config["conv1_filters"], 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(config["conv1_filters"], config["conv2_filters"], 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(config["conv2_filters"] * 8 * 8, config["hidden_size"]),
        nn.ReLU(),
        nn.Dropout(config["dropout"]),
        nn.Linear(config["hidden_size"], 10),
    )
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    
    # Data
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    train_dataset = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10("./data", train=False, download=True, transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=config["batch_size"])
    
    # Optimizer with tunable learning rate and weight decay
    optimizer = optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )
    
    criterion = nn.CrossEntropyLoss()
    
    # Training loop
    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0.0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Evaluation
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
        
        accuracy = correct / total
        avg_loss = train_loss / len(train_loader)
        
        # Report metrics to Ray Tune
        # Tune uses these to decide which trials to continue/stop
        train.report({"loss": avg_loss, "accuracy": accuracy})

## 4. Define Search Space

Ray Tune supports various sampling methods for hyperparameters.

In [ ]:
# Search space definition
search_space = {
    # Architecture hyperparameters
    "conv1_filters": tune.choice([16, 32, 64]),
    "conv2_filters": tune.choice([32, 64, 128]),
    "hidden_size": tune.choice([128, 256, 512]),
    
    # Regularization
    "dropout": tune.uniform(0.1, 0.5),
    "weight_decay": tune.loguniform(1e-5, 1e-2),
    
    # Training hyperparameters
    "lr": tune.loguniform(1e-4, 1e-2),
    "batch_size": tune.choice([32, 64, 128]),
    
    # Fixed parameters
    "epochs": 10,
}

print("Search Space:")
for key, value in search_space.items():
    print(f"  {key}: {value}")

## 5. Configure Scheduler and Search Algorithm

### ASHA Scheduler
Aggressively stops underperforming trials early, saving compute.

In [ ]:
# ASHA scheduler for early stopping
scheduler = ASHAScheduler(
    metric="accuracy",        # Metric to optimize
    mode="max",               # Maximize accuracy
    max_t=10,                 # Max epochs per trial
    grace_period=2,           # Min epochs before stopping
    reduction_factor=2,       # Halving rate
)

print("ASHA Scheduler configured:")
print("  - Stops underperforming trials early")
print("  - Focuses compute on promising trials")

In [ ]:
# Optuna search algorithm (Bayesian optimization)
# Uses TPE (Tree-structured Parzen Estimator) by default
search_alg = OptunaSearch(
    metric="accuracy",
    mode="max",
)

print("Optuna Search Algorithm configured:")
print("  - Uses Bayesian optimization")
print("  - Learns from previous trials to suggest better hyperparameters")

## 6. Run Hyperparameter Search

In [ ]:
from ray.tune import TuneConfig, RunConfig

# Tune configuration
tune_config = TuneConfig(
    scheduler=scheduler,
    search_alg=search_alg,
    num_samples=20,  # Total number of trials to run
    max_concurrent_trials=MAX_CONCURRENT_TRIALS,
)

# Run configuration
run_config = RunConfig(
    name="cifar10-hpo",
    storage_path=STORAGE_PATH,
)

print(f"Running {tune_config.num_samples} trials")
print(f"Max concurrent: {MAX_CONCURRENT_TRIALS}")

In [ ]:
# Create Tuner
tuner = tune.Tuner(
    tune.with_resources(
        train_cifar,
        resources={"cpu": 2, "gpu": 1 if NUM_GPUS > 0 else 0},
    ),
    param_space=search_space,
    tune_config=tune_config,
    run_config=run_config,
)

print("Tuner created successfully")

In [ ]:
# Run hyperparameter search
print("Starting hyperparameter search...")
print("="*50)

results = tuner.fit()

print("="*50)
print("Hyperparameter search completed!")

## 7. Analyze Results

In [ ]:
# Get best result
best_result = results.get_best_result(metric="accuracy", mode="max")

print("Best Trial:")
print(f"  Accuracy: {best_result.metrics['accuracy']:.4f}")
print(f"  Loss: {best_result.metrics['loss']:.4f}")
print("\nBest Hyperparameters:")
for key, value in best_result.config.items():
    if key != "epochs":
        print(f"  {key}: {value}")

In [ ]:
# Get results as DataFrame for analysis
results_df = results.get_dataframe()

print(f"Total trials: {len(results_df)}")
print(f"\nAccuracy statistics:")
print(results_df["accuracy"].describe())

In [ ]:
# Top 5 trials
top_5 = results_df.nlargest(5, "accuracy")[["accuracy", "loss", "config/lr", "config/hidden_size", "config/dropout"]]
print("\nTop 5 Trials:")
print(top_5.to_string())

## 8. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Plot accuracy distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy histogram
axes[0].hist(results_df["accuracy"], bins=15, edgecolor="black", alpha=0.7)
axes[0].axvline(best_result.metrics["accuracy"], color="red", linestyle="--", label="Best")
axes[0].set_xlabel("Accuracy")
axes[0].set_ylabel("Count")
axes[0].set_title("Accuracy Distribution Across Trials")
axes[0].legend()

# Learning rate vs accuracy
axes[1].scatter(results_df["config/lr"], results_df["accuracy"], alpha=0.6)
axes[1].set_xscale("log")
axes[1].set_xlabel("Learning Rate (log scale)")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Learning Rate vs Accuracy")

plt.tight_layout()
plt.show()

In [ ]:
# Hyperparameter importance (correlation with accuracy)
hp_columns = [col for col in results_df.columns if col.startswith("config/") and results_df[col].dtype in ['float64', 'int64']]

if hp_columns:
    correlations = {}
    for col in hp_columns:
        corr = results_df[col].corr(results_df["accuracy"])
        correlations[col.replace("config/", "")] = corr
    
    # Sort by absolute correlation
    sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    print("Hyperparameter Importance (correlation with accuracy):")
    for hp, corr in sorted_corr:
        print(f"  {hp}: {corr:.3f}")

## 9. Alternative: Grid Search

In [ ]:
# Grid search example (for smaller search spaces)
grid_search_space = {
    "conv1_filters": tune.grid_search([32, 64]),
    "conv2_filters": tune.grid_search([64, 128]),
    "hidden_size": 256,
    "dropout": tune.grid_search([0.2, 0.4]),
    "weight_decay": 1e-4,
    "lr": tune.grid_search([1e-3, 5e-4]),
    "batch_size": 64,
    "epochs": 5,
}

# Calculate total combinations
total_combinations = 2 * 2 * 2 * 2  # Each grid_search has 2 values
print(f"Grid search would run {total_combinations} trials")
print("Use this for exhaustive search of small spaces")

## 10. Integration with Ray Train

For distributed training within each trial, combine Tune with Train.

In [ ]:
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig

# Example: Distributed training within each HPO trial
"""
def distributed_train_func(config):
    # Your distributed training code here
    # Uses Ray Train's DDP under the hood
    pass

# Wrap with TorchTrainer for distributed execution
trainer = TorchTrainer(
    train_loop_per_worker=distributed_train_func,
    scaling_config=ScalingConfig(num_workers=2, use_gpu=True),
)

# Use trainer.as_trainable() with Tune
tuner = tune.Tuner(
    trainer.as_trainable(),
    param_space={"train_loop_config": search_space},
    tune_config=tune_config,
)
"""
print("See code cell for Ray Train + Tune integration example")

## Cleanup

In [ ]:
shutdown_ray()
print("Ray cluster shutdown complete")

## Key Takeaways

1. **Search Spaces**: Use `tune.choice`, `tune.uniform`, `tune.loguniform` for different parameter types
2. **ASHA Scheduler**: Aggressively prunes underperforming trials, saving compute
3. **Optuna Search**: Uses Bayesian optimization to explore promising regions
4. **Parallel Trials**: Run multiple trials concurrently on available GPUs
5. **Analysis**: Use DataFrames and visualizations to understand hyperparameter importance

## Next Steps

- **Population Based Training (PBT)**: Adaptive hyperparameter schedules
- **Multi-objective**: Optimize for accuracy and latency simultaneously
- **Distributed trials**: Use Ray Train for multi-GPU trials

## Resources

- [Ray Tune Documentation](https://docs.ray.io/en/latest/tune/index.html)
- [Search Algorithms](https://docs.ray.io/en/latest/tune/api/suggestion.html)
- [Schedulers](https://docs.ray.io/en/latest/tune/api/schedulers.html)
- [Tune + Train Integration](https://docs.ray.io/en/latest/train/user-guides/hyperparameter-optimization.html)